# 7.15 Attention Mechanisms

[Back to neural networks index](../7_neural_networks.ipynb)

This notebook introduces attention as a mechanism for selectively weighting information instead of compressing everything into one recurrent state.


## 7.15.1 Why recurrence struggles at scale

### Why It Matters
As sequences get longer, important signals may be many steps away from the current position. Attention provides a direct path to earlier information instead of depending only on repeated recurrence.


In [ ]:
import torch
import torch.nn.functional as F

sequence = torch.tensor([[4.0, 0.0], [0.2, 0.1], [0.1, -0.1], [0.0, 0.2]])
recurrent_summary = torch.tensor([0.0, 0.0])
for token in sequence:
    recurrent_summary = 0.6 * recurrent_summary + token

query = torch.tensor([[1.0, 0.0]])
weights = F.softmax(query @ sequence.T, dim=1)
attention_summary = weights @ sequence

{
    "recurrent_summary": recurrent_summary.round(decimals=3).tolist(),
    "attention_weights": weights.squeeze(0).round(decimals=3).tolist(),
    "attention_summary": attention_summary.squeeze(0).round(decimals=3).tolist(),
}


### Interpretation
The point is structural: attention can connect distant tokens directly instead of step by step.


## 7.15.2 Query, key, and value intuition

### Why It Matters
Attention scores come from comparing a query against keys, then using the resulting weights to mix values.


In [ ]:
import torch
import torch.nn.functional as F

query = torch.tensor([[1.0, 0.0]])
keys = torch.tensor([[1.0, 0.0], [0.2, 0.9], [-1.0, 0.0]])
values = torch.tensor([[10.0, 0.0], [0.0, 5.0], [-3.0, 1.0]])
scores = query @ keys.T
weights = F.softmax(scores, dim=1)
context = weights @ values
{"attention_weights": weights.squeeze(0).round(decimals=3).tolist(), "context_vector": context.squeeze(0).round(decimals=3).tolist()}


### Interpretation
The query is pulling information from the values through similarity with the keys.


## 7.15.3 Attention weights

### Why It Matters
Softmax turns raw compatibility scores into a probability-like weighting over source positions.


In [ ]:
import torch
import torch.nn.functional as F

scores = torch.tensor([0.2, 1.4, -0.3, 0.7])
weights = F.softmax(scores, dim=0)
{"scores": scores.tolist(), "weights": weights.round(decimals=3).tolist(), "weights_sum": round(float(weights.sum().item()), 3)}


### Interpretation
The weights tell you where the model is focusing at that moment.


## 7.15.4 Context vectors

### Why It Matters
A context vector is the weighted combination of values. Changing the attention weights changes the information passed forward.


In [ ]:
import torch

values = torch.tensor([[2.0, 0.0], [0.0, 3.0], [1.0, 1.0]])
weights_a = torch.tensor([0.8, 0.1, 0.1])
weights_b = torch.tensor([0.1, 0.8, 0.1])
context_a = weights_a @ values
context_b = weights_b @ values
{"context_focus_first": context_a.tolist(), "context_focus_second": context_b.tolist()}


### Interpretation
Attention is not only about where the model looks, but also about what combined information it passes on.


## 7.15.5 Self-attention conceptually

### Why It Matters
In self-attention, the same sequence supplies the queries, keys, and values. The mechanism asks each token which other tokens matter to it.


In [ ]:
import torch
import torch.nn.functional as F

X = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
scores = X @ X.T
weights = F.softmax(scores, dim=1)
attended = weights @ X
{"attention_matrix": weights.round(decimals=3).tolist(), "attended_vectors": attended.round(decimals=3).tolist()}


### Interpretation
Each row of the attention matrix is one token deciding how much to borrow from every token in the sequence.


## 7.15.6 A small attention example

### Why It Matters
This final subsection packages the idea into a tiny trainable attention module over token embeddings.


In [ ]:
import torch
from torch import nn

class TinyAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.query = nn.Linear(dim, dim, bias=False)
        self.key = nn.Linear(dim, dim, bias=False)
        self.value = nn.Linear(dim, dim, bias=False)
    def forward(self, x):
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)
        scores = Q @ K.transpose(-2, -1) / (x.size(-1) ** 0.5)
        weights = scores.softmax(dim=-1)
        return weights, weights @ V

torch.manual_seed(39)
x = torch.randn(2, 4, 6)
attention = TinyAttention(6)
weights, attended = attention(x)
{"weight_shape": list(weights.shape), "attended_shape": list(attended.shape), "row_sums_first_batch": weights[0].sum(dim=-1).round(decimals=3).tolist()}


### Interpretation
This is the mechanical core that later grows into full transformer blocks.
